# PINN Review — Part I: Papers 1–5 (2019–2020)

Executable reproductions for the first five foundational PINN papers:
the original formulation, uncertainty quantification methods, and adaptive sampling.

Source: Physics-Informed Neural Networks 2019–2026 review article.


## Shared Setup


In [ ]:
import math, time, random, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)
SEED=7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cpu'

def rel_l2(yhat,y):
    return float(torch.linalg.norm(yhat-y)/torch.linalg.norm(y))

class MLP(nn.Module):
    def __init__(self, din=1, dout=1, width=32, depth=3, dropout=0.0, act=nn.Tanh):
        super().__init__(); layers=[]; d=din
        for _ in range(depth):
            layers += [nn.Linear(d,width), act()]
            if dropout: layers += [nn.Dropout(dropout)]
            d=width
        layers += [nn.Linear(d,dout)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def grad(y,x,order=1):
    g=y
    for _ in range(order):
        g=torch.autograd.grad(g,x,torch.ones_like(g),create_graph=True,retain_graph=True)[0]
    return g

def train(loss_fn, model, steps=600, lr=2e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); hist=[]
    t=time.time()
    for i in range(steps):
        opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
        if i % max(1,steps//100)==0: hist.append(float(loss.detach()))
    return hist, time.time()-t

results=[]
def record(paper,method,error,seconds,notes):
    results.append(dict(paper=paper,method=method,relative_L2=error,seconds=seconds,notes=notes))


## Paper 1 — Raissi, Perdikaris & Karniadakis (2019): The Original PINN

Solves the 1D Poisson problem. This is the canonical baseline.


In [ ]:
torch.manual_seed(SEED)
base=MLP(width=28,depth=3)
xr=torch.linspace(0,1,80).reshape(-1,1).requires_grad_(True)
xb=torch.tensor([[0.0],[1.0]])
def base_loss():
    u=base(xr); uxx=grad(u,xr,2)
    return ((uxx+math.pi**2*torch.sin(math.pi*xr))**2).mean()+50*(base(xb)**2).mean()
h,t=train(base_loss,base,steps=300)
xt=torch.linspace(0,1,300).reshape(-1,1); yt=torch.sin(math.pi*xt)
with torch.no_grad(): yp=base(xt)
e=rel_l2(yp,yt); record('Raissi et al. 2019','PINN',e,t,'Poisson baseline')
plt.figure(figsize=(7,4)); plt.plot(xt,yt,label='exact'); plt.plot(xt,yp,'--',label='PINN'); plt.legend(); plt.title(f'Original PINN baseline, relative L2={e:.2e}'); plt.show()


## Paper 2 — Yang & Perdikaris (2019): Adversarial Uncertainty Quantification

GAN-style adversarial training to estimate prediction uncertainty without labeled data.


In [ ]:
torch.manual_seed(SEED)
tobs=torch.linspace(0,1,18).reshape(-1,1); ytrue=torch.exp(-1.5*tobs); yobs=ytrue+0.05*torch.randn_like(ytrue)
class LatentNet(nn.Module):
    def __init__(self): super().__init__(); self.m=MLP(2,1,28,3)
    def forward(self,t,z): return self.m(torch.cat([t,z],1))
def fit_latent(physics):
    m=LatentNet(); opt=torch.optim.Adam(m.parameters(),2e-3); st=time.time()
    for _ in range(300):
        z=torch.randn_like(tobs); tt=tobs.clone().requires_grad_(True); pred=m(tt,z)
        loss=((pred-yobs)**2).mean()
        if physics: loss=loss+0.4*((grad(pred,tt)+1.5*pred)**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return m,time.time()-st
m0,t0=fit_latent(False); mp,tp=fit_latent(True)
tg=torch.linspace(0,1,180).reshape(-1,1)
def ensemble(m):
    m.eval(); return torch.stack([m(tg,torch.randn_like(tg)).detach().squeeze() for _ in range(80)])
A=ensemble(m0); B=ensemble(mp); exact=torch.exp(-1.5*tg.squeeze())
e0=rel_l2(A.mean(0),exact); ep=rel_l2(B.mean(0),exact)
record('Yang & Perdikaris 2019','latent data-only',e0,t0,'uncertainty proxy'); record('Yang & Perdikaris 2019','physics-informed latent',ep,tp,'uncertainty proxy')
plt.figure(figsize=(7,4));
for Q,label in [(A,'data only'),(B,'physics informed')]:
    mu=Q.mean(0); sd=Q.std(0); plt.plot(tg,mu,label=label); plt.fill_between(tg.squeeze(),mu-sd,mu+sd,alpha=.15)
plt.scatter(tobs,yobs,s=18,label='noisy data'); plt.plot(tg,exact,'k--',label='exact'); plt.legend(); plt.title('Latent predictive distributions'); plt.show()


## Paper 3 — Zhang et al. (2019): Total Uncertainty with NN-aPC and Dropout

Combines Monte Carlo dropout with polynomial chaos expansion for aleatoric and epistemic UQ.


In [ ]:
torch.manual_seed(SEED)
N=500; x=torch.rand(N,1); xi=2*torch.rand(N,1)-1; y=torch.sin(math.pi*x)*(1+.25*xi)
uq=MLP(2,1,36,3,dropout=.1)
def uq_loss(): return ((uq(torch.cat([x,xi],1))-y)**2).mean()
h,t=train(uq_loss,uq,steps=300)
xg=torch.linspace(0,1,150).reshape(-1,1); uq.train()
S=[]
for _ in range(100):
    xis=2*torch.rand_like(xg)-1; S.append(uq(torch.cat([xg,xis],1)).detach().squeeze())
S=torch.stack(S); mu=S.mean(0); sd=S.std(0); mu0=torch.sin(math.pi*xg.squeeze())
e=rel_l2(mu,mu0); record('Zhang et al. 2019','NN-aPC/dropout proxy',e,t,'stochastic family')
plt.figure(figsize=(7,4)); plt.plot(xg,mu0,'k--',label='true mean'); plt.plot(xg,mu,label='learned mean'); plt.fill_between(xg.squeeze(),mu-2*sd,mu+2*sd,alpha=.2,label='predictive spread'); plt.legend(); plt.title('Parametric + approximation uncertainty'); plt.show()


## Paper 4 — Dwivedi et al. (2019): Distributed PINN

Solves two-piece discontinuous target showing how subdomains handle multi-scale problems.


In [ ]:
torch.manual_seed(SEED)
def target_piece(x): return torch.where(x<0.5,torch.sin(2*math.pi*x),torch.sin(8*math.pi*x))
xt=torch.linspace(0,1,300).reshape(-1,1); yt=target_piece(xt)
xtrain=torch.rand(120,1); ytrain=target_piece(xtrain)
g=MLP(width=20,depth=2); h,tg=train(lambda:((g(xtrain)-ytrain)**2).mean(),g,300)
l=MLP(width=16,depth=2); r=MLP(width=16,depth=2)
xl=xtrain[xtrain[:,0]<.5]; yl=target_piece(xl); xr_=xtrain[xtrain[:,0]>=.5]; yr=target_piece(xr_)
def dl():
    iface=torch.tensor([[.5]],requires_grad=True); jump=(l(iface)-r(iface))**2
    return ((l(xl)-yl)**2).mean()+((r(xr_)-yr)**2).mean()+10*jump.mean()
pars=list(l.parameters())+list(r.parameters()); opt=torch.optim.Adam(pars,2e-3); st=time.time()
for _ in range(300): opt.zero_grad(); z=dl(); z.backward(); opt.step()
td=time.time()-st
with torch.no_grad(): yg=g(xt); yd=torch.where(xt<.5,l(xt),r(xt))
eg=rel_l2(yg,yt); ed=rel_l2(yd,yt)
record('Dwivedi et al. 2019','global NN proxy',eg,tg,'representation test'); record('Dwivedi et al. 2019','DPINN proxy',ed,td,'two cells')
plt.figure(figsize=(7,4)); plt.plot(xt,yt,'k',label='target'); plt.plot(xt,yg,'--',label='global'); plt.plot(xt,yd,':',label='distributed'); plt.legend(); plt.title('Distributed representation test'); plt.show()


## Paper 5 — Lu et al. (2020): DeepXDE and Residual-Based Adaptive Refinement

RAR progressively refines collocation toward high-residual regions.


In [ ]:
def fit_poisson(points,steps=300):
    m=MLP(width=26,depth=3); x=points.clone().requires_grad_(True); b=torch.tensor([[0.],[1.]])
    def L():
        u=m(x); return ((grad(u,x,2)+math.pi**2*torch.sin(math.pi*x))**2).mean()+50*(m(b)**2).mean()
    h,t=train(L,m,steps); return m,t
xu=torch.linspace(0,1,25).reshape(-1,1); mu,tu=fit_poisson(xu)
cand=torch.linspace(0,1,400).reshape(-1,1).requires_grad_(True); res=(grad(mu(cand),cand,2)+math.pi**2*torch.sin(math.pi*cand)).abs().detach().squeeze(); add=cand.detach()[torch.topk(res,25).indices]
mr,tr=fit_poisson(torch.cat([xu,add]),650)
with torch.no_grad(): yu=mu(xt); yr=mr(xt)
eu=rel_l2(yu,torch.sin(math.pi*xt)); er=rel_l2(yr,torch.sin(math.pi*xt))
record('Lu et al. 2020 DeepXDE','uniform PINN',eu,tu,'25 points'); record('Lu et al. 2020 DeepXDE','RAR PINN',er,tr,'25+25 points')
plt.figure(figsize=(7,4)); plt.semilogy(cand.detach(),res,label='residual before refinement'); plt.scatter(add,torch.full_like(add,res.max()),s=10,label='added points'); plt.legend(); plt.title('RAR point selection'); plt.show()
